In [0]:
%run ./01_config

In [0]:
"""
10_interval_optimizer.py  —  Decision layer: interval optimizer (RQ2)

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 12.
"""

# 10 — Decision layer: interval optimizer (FR3 / RQ2)

# Implements the age-replacement cost-rate objective of Section 4.3.1 on the fitted
# Weibull distributions from notebook 09, and scores every recommendation against the
# true optimum computable from the disclosed ground truth — the distance-from-truth
# measure no operational study can report.

# Three numbers per class, three different questions:
# - predicted saving — what the fitted model believes it will save vs the incumbent cycle
# - realised saving — what the recommendation actually saves, evaluated under the true
#   parameters (the decision-value number that feeds T4)
# - regret — how much worse the recommendation is than the true optimum

# A recommendation built on imperfect parameters can still be near-optimal in cost, because
# cost-rate curves are flat near their minimum; that flatness is why parameter error and
# decision error are different things, and measuring both is the point of this notebook.

# Shared configuration from '01_config' is assumed to be in scope.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

use_project_schema()

INK, MUTED, GRID = "#1c1c1c", "#8a8a8a", "#e0e0e0"
ACCENT, WARM, GREEN, PURPLE = "#2b6cb0", "#c05621", "#2f855a", "#6b46c1"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
FIG = f"{FIGURES}/{DATASET_VERSION}_"

def emit(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}{name}.png", bbox_inches="tight")
    fig.savefig(f"{FIG}{name}.svg", bbox_inches="tight", format="svg")
    print(f"saved {FIG}{name}.png / .svg")
    plt.show(); plt.close(fig)

# Inputs: fitted parameters, costs, incumbent cycles, ground truth

for req in ["results_parameter_recovery", "gold_class_cost_params", "gold_incumbent_cycles"]:
    if not spark.catalog.tableExists(tbl(req)):
        raise RuntimeError(f"{req} missing — run the upstream notebooks first.")

rec = spark.table(tbl("results_parameter_recovery")).toPandas().set_index("equipment_class")
costs = spark.table(tbl("gold_class_cost_params")).toPandas().set_index("equipment_class")
cycles = spark.table(tbl("gold_incumbent_cycles")).toPandas().set_index("equipment_class")
fleet_n = (spark.table(tbl("silver_equipment")).groupBy("equipment_class").count()
           .toPandas().set_index("equipment_class")["count"])

# mean downtime valuation per breakdown order, per class — the criticality-weighted term
downtime = (spark.sql(f"""
    SELECT e.equipment_class, AVG(o.downtime_valuation) AS downtime_per_breakdown
    FROM {tbl('silver_order')} o
    JOIN {tbl('silver_equipment')} e ON o.equipment_id = e.equipment_id
    WHERE o.cost_type = 'BREAKDOWN'
    GROUP BY e.equipment_class
""").toPandas().set_index("equipment_class")["downtime_per_breakdown"]).fillna(0.0)

gt = None
if spark.catalog.tableExists(tbl("bronze_ground_truth_params")):
    gt = spark.table(tbl("bronze_ground_truth_params")).toPandas().set_index("EQTYP")
    for c in ["weibull_beta", "weibull_eta_days"]:
        gt[c] = gt[c].astype(float)

if rec.eta_error_pct.median() > 50:
    print("WARNING: parameter-recovery table looks like a pre-fix run "
          f"(median eta error {rec.eta_error_pct.median():.0f}%). Re-run notebook 09 first.")

print(f"{len(rec)} classes | ground truth {'loaded' if gt is not None else 'ABSENT'}")

# Cost-rate objective

# Long-run expected cost per day of preventive interval T: preventive cost weighted by the
# probability of surviving to T, plus breakdown cost (repair settlement plus the
# criticality-weighted downtime valuation) weighted by the probability of failing first,
# divided by the expected cycle length. Classes whose fitted shape does not exceed 1 are
# flagged as unsuitable for preventive replacement — no finite optimum exists — rather than
# forced to an interval.

def cost_rate(T, beta, eta, cp, cf):
    grid = np.linspace(0.0, T, 400)
    S = np.exp(-(grid / eta) ** beta)
    ST = float(np.exp(-(T / eta) ** beta))
    return (cp * ST + cf * (1.0 - ST)) / max(float(np.trapezoid(S, grid)), 1e-9)

def optimal_interval(beta, eta, cp, cf):
    """Grid search with one refinement pass. Returns (T*, cost_rate(T*))."""
    Ts = np.linspace(20.0, eta * 3.0, 300)
    r = np.array([cost_rate(t, beta, eta, cp, cf) for t in Ts])
    i = int(r.argmin())
    lo, hi = Ts[max(i - 1, 0)], Ts[min(i + 1, len(Ts) - 1)]
    Ts2 = np.linspace(lo, hi, 60)
    r2 = np.array([cost_rate(t, beta, eta, cp, cf) for t in Ts2])
    j = int(r2.argmin())
    return float(Ts2[j]), float(r2[j])

rows, curves = [], {}
for cls in sorted(rec.index):
    b_fit, e_fit = float(rec.loc[cls, "beta_fitted"]), float(rec.loc[cls, "eta_fitted"])
    cp = float(costs.loc[cls, "preventive_cost_median"])
    cf = float(costs.loc[cls, "breakdown_cost_median"]) + float(downtime.get(cls, 0.0))
    as_is = float(cycles.loc[cls, "cycle_days_mean"])

    row = {"equipment_class": cls, "beta_fitted": round(b_fit, 3),
           "cost_ratio": round(cf / cp, 2), "as_is_cycle": int(round(as_is))}

    if b_fit <= 1.0:
        row.update({"recommendation": "RUN-TO-FAILURE (no wear-out)", "T_recommended": None})
        rows.append(row); continue

    T_rec, rate_rec_fit = optimal_interval(b_fit, e_fit, cp, cf)
    rate_asis_fit = cost_rate(as_is, b_fit, e_fit, cp, cf)
    row.update({
        "T_recommended": int(round(T_rec)),
        "direction": "extend" if T_rec > as_is else "shorten",
        "predicted_saving_pct": round(100 * (rate_asis_fit - rate_rec_fit) / rate_asis_fit, 1),
    })

    if gt is not None and cls in gt.index:
        b_t, e_t = float(gt.loc[cls, "weibull_beta"]), float(gt.loc[cls, "weibull_eta_days"])
        T_true, rate_true_opt = optimal_interval(b_t, e_t, cp, cf)
        rate_asis_true = cost_rate(as_is, b_t, e_t, cp, cf)
        rate_rec_true = cost_rate(T_rec, b_t, e_t, cp, cf)
        row.update({
            "T_true_optimum": int(round(T_true)),
            "realised_saving_pct": round(100 * (rate_asis_true - rate_rec_true) / rate_asis_true, 1),
            "true_available_saving_pct": round(100 * (rate_asis_true - rate_true_opt) / rate_asis_true, 1),
            "regret_pct": round(100 * (rate_rec_true - rate_true_opt) / rate_true_opt, 1),
        })
        Ts = np.linspace(20, max(e_t, e_fit) * 2.2, 250)
        curves[cls] = (Ts,
                       np.array([cost_rate(t, b_t, e_t, cp, cf) for t in Ts]) / rate_asis_true,
                       as_is, T_rec, T_true)
    rows.append(row)

opt = pd.DataFrame(rows)
display(spark.createDataFrame(opt.astype(object).where(pd.notna(opt), None)))

# Fleet aggregate against the T4 criterion (analytic view)

# Weighted by equipment count and incumbent cost rate, so classes that spend more count
# more. This is the model-predicted and truth-evaluated *analytic* saving; the pre-registered
# T4 verdict (≥15% vs calendar PM) is confirmed by replay in notebook 12, where the same
# recommendations face simulated trajectories rather than expectations.

ok = opt.dropna(subset=["T_recommended"]).copy()
if gt is not None and "realised_saving_pct" in ok.columns:
    w = fleet_n.reindex(ok.equipment_class).values
    base = []
    for cls in ok.equipment_class:
        b_t, e_t = float(gt.loc[cls, "weibull_beta"]), float(gt.loc[cls, "weibull_eta_days"])
        cp = float(costs.loc[cls, "preventive_cost_median"])
        cf = float(costs.loc[cls, "breakdown_cost_median"]) + float(downtime.get(cls, 0.0))
        base.append(cost_rate(float(cycles.loc[cls, "cycle_days_mean"]), b_t, e_t, cp, cf))
    weight = w * np.array(base)
    fleet_realised = float(np.average(ok.realised_saving_pct, weights=weight))
    fleet_available = float(np.average(ok.true_available_saving_pct, weights=weight))
    print(f"fleet realised saving (truth-evaluated, cost-weighted): {fleet_realised:.1f}%")
    print(f"fleet available saving at the true optimum:             {fleet_available:.1f}%")
    print(f"capture ratio: {100*fleet_realised/max(fleet_available,1e-9):.0f}% of available savings")
    print(f"directions: {int((ok.direction=='shorten').sum())} shorten / "
          f"{int((ok.direction=='extend').sum())} extend")
    print(f"\nT4 analytic indication (criterion >=15%): "
          f"{'MET' if fleet_realised >= 15 else 'NOT MET'} at {fleet_realised:.1f}% "
          f"(replay confirmation in notebook 12)")

# Figure O1 — cost-rate curves with as-is, recommended and true-optimal intervals

if curves:
    n = len(curves)
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(12.5, 5.8), sharey=False)
    for ax, (cls, (Ts, rel, as_is, T_rec, T_true)) in zip(np.atleast_1d(axes).ravel(), curves.items()):
        ax.plot(Ts, rel, color=ACCENT, lw=1.5)
        ax.axvline(as_is, color=MUTED, ls="--", lw=1.1)
        ax.axvline(T_rec, color=WARM, lw=1.3)
        ax.axvline(T_true, color=GREEN, ls=":", lw=1.5)
        ax.set_title(cls.replace("-", "\n"), fontsize=7.5)
        ax.tick_params(labelsize=7)
        ax.set_ylim(bottom=min(0.5, rel.min() * 0.95))
    for ax in np.atleast_1d(axes).ravel()[n:]:
        ax.set_visible(False)
    np.atleast_1d(axes).ravel()[0].set_ylabel("true cost rate (as-is = 1.0)")
    fig.suptitle("grey dashed = incumbent   orange = recommended (fitted)   green dotted = true optimum",
                 fontsize=8, color=MUTED, y=1.02)
    emit(fig, "o1_cost_rate_recommendations")

# Figure O2 — realised savings by class and direction

if gt is not None and len(ok):
    d = ok.sort_values("realised_saving_pct")
    fig, ax = plt.subplots(figsize=(8, 4.6))
    cols = [PURPLE if x == "shorten" else ACCENT for x in d.direction]
    ax.barh(d.equipment_class, d.realised_saving_pct, color=cols, height=.68)
    ax.scatter(d.true_available_saving_pct, np.arange(len(d)), color=GREEN, s=42,
               zorder=3, label="available at true optimum")
    ax.axvline(0, color=MUTED, lw=.8)
    ax.set_xlabel("cost-rate saving vs incumbent cycle, evaluated under true parameters (%)")
    handles = [plt.Rectangle((0, 0), 1, 1, color=ACCENT), plt.Rectangle((0, 0), 1, 1, color=PURPLE)]
    ax.legend(handles + [plt.Line2D([], [], color=GREEN, marker='o', ls='')],
              ["extend", "shorten", "available at true optimum"],
              frameon=False, loc="lower right", fontsize=8)
    emit(fig, "o2_realised_savings")

# Persist

from pyspark.sql.functions import lit
(spark.createDataFrame(opt.astype(object).where(pd.notna(opt), None))
      .withColumn("dataset_version", lit(DATASET_VERSION))
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(tbl("results_interval_optimizer")))
opt.to_csv(f"{EXPORTS}/interval_optimizer.csv", index=False)
print("results_interval_optimizer written; exported to exports/")
print("\nSection 3.4.1 table: use interval_optimizer.csv. The two-directional split and the "
      "regret column are the substantive findings; regret near zero despite scale error is the "
      "flat-minimum effect and worth a sentence in the write-up.")